# Implicit Functions and Automatic Differentiation

Let $z^\star$ solve

$$
F(z^\star,\theta)=f_\theta(z^\star,x)-z^\star=0.
$$

Differentiate $F=0$:

$$
\frac{\partial f_\theta}{\partial z}\,dz
+\frac{\partial f_\theta}{\partial \theta}\,d\theta
-dz=0.
$$

Collect the $dz$ terms:

$$
(I-J_f)\,dz=\frac{\partial f_\theta}{\partial \theta}\,d\theta.
$$

Reverse-mode differentiation solves the transposed linear system

$$
(I-J_f^\top)u=g,
$$

where $g=\partial \ell/\partial z^\star$.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

from silva_networks import resolve_device, SolverConfig

torch.manual_seed(7)
np.random.seed(7)
device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
device

## Materialized Jacobian on a Small State

For a tiny state it is useful to materialize $J_f$. This makes the
matrix-free `vjp` and `jvp` calls easy to verify.

In [ ]:
from silva_networks import (
    SILVAImplicitTransition,
    fixed_point,
    full_jacobian,
    vjp,
    jvp,
    implicit_adjoint_solve,
)

transition = SILVAImplicitTransition(2, 2, spectral_scale=0.35).to(device)
x = torch.tensor([[0.3, -0.5]], device=device)
z0 = torch.zeros(1, 2, device=device)

def f(z):
    return transition(z, x)

solve = fixed_point(f, z0, SolverConfig(max_iter=20, alpha=0.7))
J = full_jacobian(f, solve.z)
J

In [ ]:
probe = torch.randn_like(solve.z)
_, jvp_value = jvp(f, solve.z, probe)
vjp_value = vjp(f, solve.z, probe)

checks = {
    "Jv_close": torch.allclose(J @ probe.reshape(-1), jvp_value.reshape(-1), atol=1e-5),
    "JT_v_close": torch.allclose(J.T @ probe.reshape(-1), vjp_value.reshape(-1), atol=1e-5),
}
checks

## Adjoint Solve

The explicit matrix solution is

$$
u=(I-J_f^\top)^{-1}g.
$$

The package helper computes the same object with VJP-backed GMRES.

In [ ]:
g = torch.tensor([[1.0, -0.25]], device=device)
explicit_u = torch.linalg.solve(torch.eye(2, device=device) - J.T, g.reshape(-1)).reshape_as(g)
gmres_u = implicit_adjoint_solve(f, solve.z, g, max_iter=8, tol=1e-7)

explicit_u.detach().cpu().numpy(), gmres_u.x.detach().cpu().numpy(), gmres_u.residuals

## Jacobian Spectrum

The spectral radius $\rho(J_f)$ is a local fixed-point diagnostic. A value
below one is compatible with local contraction. A value near or above one means
the solver may need damping, a different solver, or regularization.

In [ ]:
from silva_networks import spectral_radius, hutchinson_jacobian_norm

rho = spectral_radius(f, solve.z, iters=12)
fro = hutchinson_jacobian_norm(f, solve.z, samples=2, squared=False)
float(rho), float(fro.detach().cpu())

## Citation and Sources

If this package or notebook is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
https://doi.org/10.5281/zenodo.21770099
```

When the work is connected to the SILVA methodology, cite the SILVA Networks
paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background sources:

- Deep Implicit Layers tutorial: https://implicit-layers-tutorial.org/
- LocusLab DEQ repository: https://github.com/locuslab/deq
- Deep Equilibrium Models: https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models: https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization: https://arxiv.org/abs/2106.14342

The notebook is adapted to the `silva_networks` public API. It links to the
sources above and keep the examples package-native.

## Where to Go Next

| Question | Page |
| --- | --- |
| How is the backward linear system implemented? | [Implicit Backward Guide](https://jseluis.github.io/silva-networks/learn/implicit-backward-guide/) |
| Where is implicit differentiation derived? | [Mathematical Foundations](https://jseluis.github.io/silva-networks/learn/mathematical-foundations/#implicit-differentiation) |
| Which engine controls expose backward solving? | [DEQ Engine API](https://jseluis.github.io/silva-networks/api/deq-engine/) |
